# Đối chiếu kết quả tự kiểm tra và rubric bằng Python

Mở **Kiểm tra và dự án** trên web. Làm mốc ngày 7/16/21, ghi minh chứng rubric, rồi bấm **Xuất tiến độ**. Lưu JSON cùng thư mục notebook và điền PROGRESS_PATH.

Điểm trắc nghiệm được tính từ đáp án đã nộp, không tin điểm ghi sẵn trong JSON. Rubric là **tự đánh giá**; notebook chỉ kiểm tra quy tắc cộng điểm, không kiểm chứng chất lượng minh chứng. Mốc trắc nghiệm đạt từ 6/8 khi trả lời đủ; rubric cần đủ sáu tiêu chí và từ 70/100.

Không có JSON thì notebook vẫn chạy với trạng thái trống; không tạo thành tích giả và không sửa tiến độ web.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
# Answer keys của bản học liệu này; mỗi tuple gồm id, lựa chọn đúng (bắt đầu 0).
KEYS={
 '7':list(zip(['7-price','7-mse','7-update','7-scale','7-split','7-rate','7-features','7-target'],[1,2,0,2,1,2,0,1])),
 '16':list(zip(['16-recall','16-imbalance','16-knn','16-gini','16-gain','16-validation','16-leak','16-baseline'],[1,2,0,1,2,0,2,1])),
 '21':list(zip(['21-forward','21-bce','21-dout','21-hidden','21-update','21-xor','21-old','21-threshold'],[0,2,1,0,2,1,2,0]))}
WEIGHTS={'problem':15,'data':20,'baseline':15,'evaluation':20,'reproduce':15,'explain':15}
def grade(answers,key):
    valid=lambda v:type(v) is int and 0<=v<3
    answered=sum(valid(answers.get(q)) for q,_ in key)
    correct=sum(valid(answers.get(q)) and answers[q]==right for q,right in key)
    return {'answered':answered,'correct':correct,'total':8,'passed':answered==8 and correct>=6}
def evaluate_rubric(state):
    rows=[]
    for criterion,weight in WEIGHTS.items():
        entry=state.get('rubric',{}).get(criterion,{})
        level=entry.get('level');evidence=entry.get('evidence','')
        rated=type(level) is int and 0<=level<=4
        documented=rated and (level==0 or isinstance(evidence,str) and bool(evidence.strip()))
        rows.append({'criterion':criterion,'level':level,'weight':weight,'documented':documented,'points':weight*level/4 if documented else 0})
    score=sum(r['points'] for r in rows);complete=all(r['documented'] for r in rows)
    return rows,score,complete,complete and score>=70
PROGRESS_PATH='' # ví dụ 'Tien_do_AI_30_ngay.json'
if PROGRESS_PATH:document=json.loads(Path(PROGRESS_PATH).read_text())
else:document={'version':1,'days':{},'assessment':{}};print('Chưa chọn JSON; đang hiển thị trạng thái trống.')
assert document['version']==1
state=document.get('assessment',{})

## 1. Điểm từ lần nộp gần nhất

Draft là bản đang sửa; attempts là các lần đã nộp. Chỉ lấy attempts[-1] để đối chiếu dòng “Lần nộp gần nhất” trên web. Không dùng draft thay thế bài đã nộp.

In [ ]:
rows=[]
for day,key in KEYS.items():
    attempts=state.get('checkpoints',{}).get(day,{}).get('attempts',[])
    if not attempts:rows.append({'day':day,'submitted':False,'correct':0,'passed':False})
    else:rows.append({'day':day,'submitted':True,**grade(attempts[-1]['answers'],key)})
display(pd.DataFrame(rows))
fig,ax=plt.subplots(figsize=(7,3));ax.bar([r['day'] for r in rows],[r['correct'] for r in rows],color='#277d9b');ax.axhline(6,color='#856212',linestyle='--',label='Threshold 6/8');ax.set(ylim=(0,8.5),xlabel='Day',ylabel='Correct answers',title='Latest submissions (no submission shown as zero)');ax.legend();plt.tight_layout();plt.show()

## 2. Điểm rubric phải có mức và minh chứng

Điểm = trọng số × mức/4. Mức 0 là đánh giá tường minh và có thể không có minh chứng; mức 1–4 thiếu minh chứng bị tính 0 và chưa đủ tiêu chí. Tổng 85 nhưng thiếu một tiêu chí vẫn chưa đủ điều kiện. Chuỗi minh chứng có nội dung không đồng nghĩa đã được thẩm định.

In [ ]:
rubric_rows,score,complete,ready=evaluate_rubric(state)
display(pd.DataFrame(rubric_rows))
print('Tổng có minh chứng:',score,'/100; đủ tiêu chí:',complete,'; đạt ngưỡng tự đánh giá:',ready)
fig,ax=plt.subplots(figsize=(9,3));ax.bar([r['criterion'] for r in rubric_rows],[r['points'] for r in rubric_rows],label='Evidence-counted points',color='#397750');ax.scatter(range(6),list(WEIGHTS.values()),color='#856212',label='Maximum');ax.set(ylabel='Points',title='Self assessment: evidence content is not independently verified');ax.legend();plt.xticks(rotation=20);plt.tight_layout();plt.show()

## 3. Tính tay và kiểm tra các biên

A. Tính điểm khi một tiêu chí trọng số 20 có mức 3; sau đó xóa minh chứng và dự đoán điểm.

B. Tạo trường hợp đúng 6/8, 5/8, còn thiếu một câu; giải thích vì sao chưa trả lời đủ không được đạt dù đã đúng 6.

C. Dùng notebook ngày 28–30 cho dự án riêng: ghi X/y, nguồn, train/validation/test, baseline, thí nghiệm, lỗi và giới hạn. Restart & Run All trước khi tự đánh giá mức tái lập. Trên web mở rubric để xem mô tả chi tiết từng mức 0–4.

In [ ]:
key=KEYS['7'];all_correct=dict(key)
assert grade(all_correct,key)['correct']==8
six=dict(all_correct)
for q,right in key[:2]:six[q]=(right+1)%3
assert grade(six,key)['passed']
partial=dict(key[:6]);assert grade(partial,key)['correct']==6 and not grade(partial,key)['passed']
full={'rubric':{k:{'level':4,'evidence':'Example evidence, not a learner submission'} for k in WEIGHTS}}
assert evaluate_rubric(full)[1:]==(100,True,True)
full['rubric']['problem']['evidence']=''
assert evaluate_rubric(full)[1:]==(85,False,False)
assert np.isclose(.4-.1*(-.2),.42)
print('Các trường hợp biên khớp. Đây là kiểm tra quy tắc, không sửa tiến độ của em.')